# PetGuard AI — Model & Zone Evaluation

**Midterm dataset plan:** Oxford-IIIT Pets (100), COCO 2017 validation (50), and PetGuard-specific fixed-camera samples (50).

**Evaluation, not training.** Compare YOLOv8n and YOLOv8s using the same labeled samples. Select model/threshold on a development subset, then report held-out results. COCO is a benchmark used in pretrained-model development, not a fully independent source.

The public datasets primarily measure **object-level pet detection**. Your own labeled videos measure **zone decisions**. Do not combine these denominators.


## 1. Clone the repository and install evaluation dependencies

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = Path("/content/PetGuard-AI")
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "--branch", "master",
                    "https://github.com/iamhuongthn/PetGuard-AI.git", str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(REPO/"requirements.txt")], check=True)
sys.path.insert(0, str(REPO))
from src.evaluate import match_detections, detection_metrics, evaluate_zones
from ultralytics import YOLO
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
import json, time, tarfile, urllib.request
DATA = REPO/"data"/"downloads"
DATA.mkdir(parents=True, exist_ok=True)
METRICS = REPO/"results"/"metrics"
METRICS.mkdir(parents=True, exist_ok=True)
print("Evaluation environment ready")


## 2. Oxford-IIIT Pets — download images + segmentation masks

The official archives are relatively large (~800 MB images plus annotations). Run this only when needed; the cell downloads and extracts them to Colab temporary storage. Oxford trimaps encode pet pixels as labels **1 or 2**, background as **3**. Convert the foreground mask to an approximate bounding box and visually inspect a sample before evaluating.

Official source: https://www.robots.ox.ac.uk/~vgg/data/pets/


In [ ]:
OXFORD = DATA/"oxford"
OXFORD.mkdir(exist_ok=True)
archives = {
    "images.tar.gz": "https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz",
    "annotations.tar.gz": "https://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz",
}
for filename, url in archives.items():
    archive = OXFORD/filename
    if not archive.exists():
        print("Downloading:", filename)
        urllib.request.urlretrieve(url, archive)
    marker = OXFORD/(filename + ".extracted")
    if not marker.exists():
        with tarfile.open(archive, "r:gz") as tar:
            # Archives are from the official dataset host; extract inside the data directory.
            tar.extractall(OXFORD, filter="data")
        marker.touch()
print("Oxford files available.")


In [ ]:
# Build a reproducible 50-cat / 50-dog evaluation manifest.
# Oxford breed names: first letter uppercase = cat; lowercase = dog.
import re
images = sorted((OXFORD/"images").glob("*.jpg"))
cats = [p for p in images if p.name[0].isupper()]
dogs = [p for p in images if p.name[0].islower()]
rng = np.random.default_rng(1378)
selected = [cats[i] for i in rng.choice(len(cats), 50, replace=False)] + [
    dogs[i] for i in rng.choice(len(dogs), 50, replace=False)]
oxford_rows = []
for path in selected:
    mask_path = OXFORD/"annotations"/"trimaps"/(path.stem+".png")
    mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)
    if mask is None:
        raise FileNotFoundError(mask_path)
    yy, xx = np.where((mask == 1) | (mask == 2))
    if len(xx) == 0:
        raise ValueError(f"Empty pet mask: {mask_path}")
    oxford_rows.append({
        "sample_id": path.stem, "image_path": str(path),
        "class": "cat" if path.name[0].isupper() else "dog",
        "bbox": [int(xx.min()), int(yy.min()), int(xx.max()+1), int(yy.max()+1)]
    })
pd.DataFrame(oxford_rows).to_json(DATA/"oxford_manifest.json", orient="records", indent=2)
print("Oxford samples:", len(oxford_rows))


## 3. COCO 2017 validation — 50 annotated pet scenes

COCO's 2017 instance annotation archive is large. This section downloads the annotations and only the **50 selected images**, not the entire validation image archive. Use COCO's provided `bbox` annotations and `coco_url` to download each image. Review failed downloads and annotation coverage.

Official source: https://cocodataset.org/#download


In [ ]:
COCO = DATA/"coco"
COCO.mkdir(exist_ok=True)
annotations_zip = COCO/"annotations_trainval2017.zip"
if not annotations_zip.exists():
    urllib.request.urlretrieve(
        "https://images.cocodataset.org/annotations/annotations_trainval2017.zip",
        annotations_zip)
import zipfile
ann_file = COCO/"annotations"/"instances_val2017.json"
if not ann_file.exists():
    with zipfile.ZipFile(annotations_zip) as z:
        z.extract("annotations/instances_val2017.json", COCO)
ann = json.loads(ann_file.read_text())
images_by_id = {i["id"]: i for i in ann["images"]}
pet_anns = {}
for a in ann["annotations"]:
    if a["category_id"] in (17, 18) and not a.get("iscrowd", 0):
        pet_anns.setdefault(a["image_id"], []).append(a)
rng = np.random.default_rng(1378)
ids = sorted(pet_anns)
selected_ids = [ids[i] for i in rng.choice(len(ids), 50, replace=False)]
coco_rows = []
for image_id in selected_ids:
    meta = images_by_id[image_id]
    image_path = COCO/meta["file_name"]
    if not image_path.exists():
        urllib.request.urlretrieve(meta["coco_url"].replace("http://", "https://"), image_path)
    boxes = []
    for a in pet_anns[image_id]:
        x, y, w, h = a["bbox"]
        boxes.append({"class": "cat" if a["category_id"] == 17 else "dog",
                      "bbox": [x, y, x+w, y+h]})
    coco_rows.append({"sample_id": str(image_id), "image_path": str(image_path),
                      "objects": boxes})
(DATA/"coco_manifest.json").write_text(json.dumps(coco_rows, indent=2))
print("COCO scenes:", len(coco_rows), "pets:", sum(len(r["objects"]) for r in coco_rows))


## 4. Run YOLOv8n and YOLOv8s on identical images

Object matching is class-aware, one-to-one, and requires IoU ≥0.5. Count unmatched predictions as false positives. This also catches human-as-dog errors when they occur in the evaluated images.

Do not tune the confidence threshold on the final held-out evaluation set.


In [ ]:
samples = [
    {"sample_id": "oxford:"+r["sample_id"], "image_path": r["image_path"],
     "objects": [{"class": r["class"], "bbox": r["bbox"]}]} for r in oxford_rows
] + [
    {"sample_id": "coco:"+r["sample_id"], "image_path": r["image_path"],
     "objects": r["objects"]} for r in coco_rows
]
rows = []
for model_name in ("yolov8n.pt", "yolov8s.pt"):
    detector = YOLO(model_name)
    for sample in samples:
        image = cv2.imread(sample["image_path"])
        if image is None:
            raise RuntimeError(f"Cannot read {sample['image_path']}")
        t0 = time.perf_counter()
        pred = detector.predict(image, classes=[15,16], conf=0.35,
                                imgsz=960, verbose=False)[0]
        elapsed_ms = (time.perf_counter()-t0)*1000
        boxes = [{"class": pred.names[int(b.cls[0])],
                  "bbox": b.xyxy[0].cpu().tolist(),
                  "confidence": float(b.conf[0])} for b in pred.boxes]
        matched = match_detections(sample["objects"], boxes, iou_threshold=0.5)
        rows.append({"model": model_name, "sample_id": sample["sample_id"],
                     **matched, "latency_ms": elapsed_ms})
    print("Completed:", model_name)
raw = pd.DataFrame(rows)
raw.to_csv(METRICS/"detection_per_image.csv", index=False)
summary = []
for model_name, group in raw.groupby("model"):
    m = detection_metrics(group.to_dict("records"))
    summary.append({"model": model_name, **m,
                    "mean_latency_ms": group["latency_ms"].mean(),
                    "p95_latency_ms": group["latency_ms"].quantile(.95)})
comparison = pd.DataFrame(summary)
comparison.to_csv(METRICS/"model_comparison.csv", index=False)
display(comparison)


### Optional helper — prepare a frame-labeling sheet from an existing Gradio run

Set `RUN_DIR` to one `results/runs/<run_id>` folder and run the cell. It samples up to 10 separated frames **per run** and exports a draft labeling CSV. Combine drafts from several independent videos to reach approximately 50 PetGuard-specific samples.

**Ground-truth labels remain blank on purpose.** Watch the original video and label SAFE/VIOLATION yourself. The predicted status comes from the system's temporally confirmed frame status. Do not evaluate the same clips used to tune your polygon or model threshold.


In [ ]:
# Optional: prepare a review sheet; do not fabricate ground-truth labels.
RUN_DIR = None  # e.g., REPO/"results"/"runs"/"your_run_id"
if RUN_DIR is None:
    print("Set RUN_DIR to an actual Gradio run folder to create a draft labeling sheet.")
else:
    run_dir = Path(RUN_DIR)
    predictions = pd.read_csv(run_dir/"frame_predictions.csv")
    sample_count = min(10, len(predictions))
    indices = np.linspace(0, len(predictions)-1, sample_count, dtype=int)
    sampled = predictions.iloc[indices].copy()
    sampled["sample_id"] = [f"{run_dir.name}_frame_{int(f)}" for f in sampled["frame"]]
    sampled["source"] = "PetGuard video"
    sampled["video_id"] = run_dir.name
    sampled["frame_id"] = sampled["frame"]
    sampled["scenario"] = ""
    sampled["pet_count_ground_truth"] = pd.NA
    sampled["expected_zone_status"] = ""  # manually label before computing metrics
    sampled["predicted_zone_status"] = sampled["confirmed_violation"].map(
        {True:"VIOLATION", False:"SAFE"})
    sampled["zone_mode"] = json.loads((run_dir/"zone_config.json").read_text())["zone_mode"]
    sampled["notes"] = ""
    cols = ["sample_id","source","video_id","frame_id","scenario",
            "pet_count_ground_truth","expected_zone_status",
            "predicted_zone_status","zone_mode","notes"]
    draft = run_dir/"zone_labels_DRAFT.csv"
    sampled[cols].to_csv(draft, index=False)
    print("Draft created:", draft, "— ground truth must be filled manually.")


## 5. PetGuard-specific zone evaluation — 50 independently labeled samples

Create `data/evaluation_labels.csv` using the provided template. Include source video and frame IDs, the confirmed zone used, ground-truth SAFE/VIOLATION, and the predicted SAFE/VIOLATION. To avoid leakage, label before looking at predictions and avoid treating adjacent near-identical frames as independent.

For **frame-level** evaluation, consistently use `candidate_violation` (raw zone decision) or `confirmed_violation` (temporally filtered status), and state which. A video alert is an event, not a frame-level label.

Human-only and empty-room frames should have expected status SAFE. Do not fabricate labels or fill the template with example counts.


In [ ]:
LABELS = REPO/"data"/"evaluation_labels.csv"
if not LABELS.exists():
    print("Not evaluated yet. Create data/evaluation_labels.csv with verified labels.")
else:
    metrics = evaluate_zones(LABELS, METRICS/"zone_metrics.csv")
    display(pd.DataFrame([metrics]))
    matrix = np.array([[metrics["TN"],metrics["FP"]],
                       [metrics["FN"],metrics["TP"]]])
    fig, ax = plt.subplots(figsize=(5,4))
    ax.imshow(matrix, cmap="Blues")
    for (i,j), v in np.ndenumerate(matrix):
        ax.text(j, i, str(v), ha="center", va="center")
    ax.set_xticks([0,1], ["SAFE","VIOLATION"])
    ax.set_yticks([0,1], ["SAFE","VIOLATION"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title("PetGuard Zone Confusion Matrix")
    plt.tight_layout()
    plt.savefig(METRICS/"confusion_matrix.png", dpi=150)
    plt.show()


## 6. Report results vs Blueprint targets

- Pet detection recall target: ≥85%.
- Zone decision accuracy target: ≥90%.
- False violation rate target: ≤10%.
- Processing: report mean/P95 milliseconds per frame, and end-to-end FPS from actual Gradio runs.

**Caveats:** Oxford mask-derived boxes are approximations; COCO is a pretrained-model benchmark; 50 zone samples may be correlated if taken from the same video. Report these limitations honestly.

**Commit only:** notebook, code, small evaluation manifests/labels, CSV metrics, and plots. Avoid committing entire Oxford/COCO archives or unlicensed videos.
